In [ ]:
from google.colab import files

uploaded = files.upload()

Saving cafa-6-protein-function-prediction.zip to cafa-6-protein-function-prediction.zip


In [ ]:
import zipfile

with zipfile.ZipFile("cafa-6-protein-function-prediction.zip", "r") as zip_ref:
    zip_ref.extractall("/content/cafa6_data")

In [ ]:
import os

print(os.listdir("/content/cafa6_data"))

['sample_submission.tsv', 'IA.tsv', 'Test', 'Train']


In [ ]:
print("Train folder contents:")
print(os.listdir("/content/cafa6_data/Train"))

print("\nTest folder contents:")
print(os.listdir("/content/cafa6_data/Test"))

Train folder contents:
['train_sequences.fasta', 'go-basic.obo', 'train_taxonomy.tsv', 'train_terms.tsv']

Test folder contents:
['testsuperset.fasta', 'testsuperset-taxon-list.tsv']


In [ ]:
path = "/content/cafa6_data/sample_submission.tsv"

with open(path, "r") as f:
    for i in range(5):
        print(repr(f.readline()))

'A0A0C5B5G6\tGO:0000001\t0.123\n'
'A0A0C5B5G6\tGO:0000002\t0.456\n'
'A0A0C5B5G6\tText\t0.123\tRegulates insulin sensitivity and metabolic homeostasis\n'
'A0A0C5B5G6\tText\t0.456\tInhibits the folate cycle, thereby reducing de novo purine biosynthesis which leads to the accumulation of the de novo purine synthesis intermediate 5-aminoimidazole-4-carboxamide (AICAR)\n'
'A0A0C5B5G6\tText\t0.456\tand the activation of the metabolic regulator 5-AMP-activated protein kinase (AMPK)\n'


In [ ]:
import csv

with open('cafa6_data/sample_submission.tsv') as f:
  reader = csv.reader(f, delimiter='\t')
  for i, row in enumerate(reader):
    print(len(row), row)
    if i > 15:
      break

3 ['A0A0C5B5G6', 'GO:0000001', '0.123']
3 ['A0A0C5B5G6', 'GO:0000002', '0.456']
4 ['A0A0C5B5G6', 'Text', '0.123', 'Regulates insulin sensitivity and metabolic homeostasis']
4 ['A0A0C5B5G6', 'Text', '0.456', 'Inhibits the folate cycle, thereby reducing de novo purine biosynthesis which leads to the accumulation of the de novo purine synthesis intermediate 5-aminoimidazole-4-carboxamide (AICAR)']
4 ['A0A0C5B5G6', 'Text', '0.456', 'and the activation of the metabolic regulator 5-AMP-activated protein kinase (AMPK)']
3 ['A0A1B0GTW7', 'GO:0000001', '0.123']
3 ['A0A1B0GTW7', 'GO:0000002', '0.456']
4 ['A0A1B0GTW7', 'Text', '0.123', 'Putative metalloproteinase that plays a role in left-right patterning process']
3 ['A0JNW5', 'GO:0000001', '0.123']
3 ['A0JNW5', 'GO:0000002', '0.456']
3 ['A0JP26', 'GO:0000001', '0.123']
3 ['A0JP26', 'GO:0000002', '0.456']
3 ['A0PK11', 'GO:0000001', '0.123']
3 ['A0PK11', 'GO:0000002', '0.456']
3 ['A1A4S6', 'GO:0000001', '0.123']
3 ['A1A4S6', 'GO:0000002', '0.456'

In [ ]:
!pip install biopython -q
from Bio import SeqIO

sequences = {}
for record in SeqIO.parse('cafa6_data/Train/train_sequences.fasta', 'fasta'):
  sequences[record.id] = str(record.seq)

print(f"Number of proteins: {len(sequences)}")
example_id = list(sequences.keys())[0]
print(example_id, sequences[example_id][:50])

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 55.6 MB/s eta 0:00:00
Number of proteins: 82404
sp|A0A0C5B5G6|MOTSC_HUMAN MRWQEMGYIFYPRKLR


In [ ]:
sequences = {}
for record in SeqIO.parse('cafa6_data/Train/train_sequences.fasta', 'fasta'):
    accession = record.id.split('|')[1]
    sequences[accession] = str(record.seq)

print(f"Number of proteins: {len(sequences)}")
example_id = list(sequences.keys())[0]
print(example_id, sequences[example_id][:50])

Number of proteins: 82404
A0A0C5B5G6 MRWQEMGYIFYPRKLR


In [ ]:
import pandas as pd

labels_df = pd.read_csv('cafa6_data/Train/train_terms.tsv', sep='\t')
print(labels_df.head())
print(labels_df.shape)
print(labels_df.columns.tolist())


  EntryID        term aspect
0  Q5W0B1  GO:0000785      C
1  Q5W0B1  GO:0004842      F
2  Q5W0B1  GO:0051865      P
3  Q5W0B1  GO:0006275      P
4  Q5W0B1  GO:0006513      P
(537027, 3)
['EntryID', 'term', 'aspect']


In [ ]:
seq_ids = set(sequences.keys())
label_ids = set(labels_df['EntryID'])

print(f"Proteins with sequences: {len(seq_ids)}")
print(f"Proteins with labels: {len(label_ids)}")
print(f"Overlap: {len(seq_ids & label_ids)}")

Proteins with sequences: 82404
Proteins with labels: 82404
Overlap: 82404


In [ ]:
print(labels_df['aspect'].value_counts())

aspect
P    250805
C    157770
F    128452
Name: count, dtype: int64


In [ ]:
!pip install goatools -q

from goatools.obo_parser import GODag

go_dag = GODag('cafa6_data/Train/go-basic.obo')

print(f"Total GO terms in DAG: {len(go_dag)}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 77.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 17.7 MB/s eta 0:00:00
cafa6_data/Train/go-basic.obo: fmt(1.2) rel(2025-06-01) 43,448 Terms
Total GO terms in DAG: 43448


In [ ]:
example_term = go_dag['GO:0004672']
print(example_term.name)
print(example_term.namespace)
print("Parents:", [p.id for p in example_term.parents])


protein kinase activity
molecular_function
Parents: ['GO:0140096', 'GO:0016773', 'GO:0016301']


In [ ]:
from collections import defaultdict

def propagate_terms(protein_terms, go_dag):
    """protein_terms: set of GO term IDs for one protein -> returns expanded set including all ancestors"""
    propagated = set(protein_terms)
    for term in protein_terms:
        if term in go_dag:
            propagated.update(go_dag[term].get_all_parents())
            propagated.add(term)
    return propagated

# Group raw labels by protein
protein_to_terms = defaultdict(set)
for _, row in labels_df.iterrows():
    protein_to_terms[row['EntryID']].add(row['term'])

# Propagate for every protein
propagated_protein_terms = {}
for protein_id, terms in protein_to_terms.items():
    propagated_protein_terms[protein_id] = propagate_terms(terms, go_dag)

# Check expansion on one example
example_protein = list(propagated_protein_terms.keys())[0]
print(f"Raw terms: {len(protein_to_terms[example_protein])}")
print(f"Propagated terms: {len(propagated_protein_terms[example_protein])}")

Raw terms: 7
Propagated terms: 37


In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer

protein_ids_ordered = list(propagated_protein_terms.keys())
term_sets_ordered = [propagated_protein_terms[pid] for pid in protein_ids_ordered]

mlb = MultiLabelBinarizer()
Y = mlb.fit_transform(term_sets_ordered)

print(f"Label matrix shape: {Y.shape}")  # (num_proteins, num_unique_GO_terms)
print(f"Example protein's label vector sum: {Y[0].sum()}")  # should match propagated count for that protein

Label matrix shape: (82404, 29175)
Example protein's label vector sum: 37


In [ ]:
import numpy as np

term_frequencies = Y.sum(axis=0)  # how many proteins have each term
print(f"Min frequency: {term_frequencies.min()}")
print(f"Max frequency: {term_frequencies.max()}")
print(f"Median frequency: {np.median(term_frequencies)}")

# How many terms survive at different thresholds
for threshold in [10, 25, 50, 100]:
    n_survive = (term_frequencies >= threshold).sum()
    print(f"Terms with >= {threshold} proteins: {n_survive}")


Min frequency: 1
Max frequency: 60292
Median frequency: 7.0
Terms with >= 10 proteins: 12734
Terms with >= 25 proteins: 7589
Terms with >= 50 proteins: 4828
Terms with >= 100 proteins: 2922


In [ ]:
keep_mask = term_frequencies >= 50
kept_terms = mlb.classes_[keep_mask]

Y_filtered = Y[:, keep_mask]

print(f"Filtered label matrix shape: {Y_filtered.shape}")
print(f"Terms kept: {len(kept_terms)}")

Filtered label matrix shape: (82404, 4828)
Terms kept: 4828


In [ ]:
proteins_with_zero_labels = (Y_filtered.sum(axis=1) == 0).sum()
print(f"Proteins with zero labels after filtering: {proteins_with_zero_labels}")


Proteins with zero labels after filtering: 0


In [ ]:
import gc

del Y, labels_df, protein_to_terms, propagated_protein_terms, go_dag
gc.collect()

149957

In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

True
Tesla T4


In [ ]:
import pickle

with open('/content/checkpoint_step2.pkl', 'wb') as f:
    pickle.dump({
        'Y_filtered': Y_filtered,
        'kept_terms': kept_terms,
        'sequences': sequences
    }, f)


In [ ]:
!pip install fair-esm -q
import esm
import torch

model, alphabet = esm.pretrained.esm2_t12_35M_UR50D()
batch_converter = alphabet.get_batch_converter()
model.eval()
model = model.to('cuda')

print("Model loaded")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.1/93.1 kB 5.3 MB/s eta 0:00:00
Downloading: "https://dl.fbaipublicfiles.com/fair-esm/models/esm2_t12_35M_UR50D.pt" to /root/.cache/torch/hub/checkpoints/esm2_t12_35M_UR50D.pt
Downloading: "https://dl.fbaipublicfiles.com/fair-esm/regression/esm2_t12_35M_UR50D-contact-regression.pt" to /root/.cache/torch/hub/checkpoints/esm2_t12_35M_UR50D-contact-regression.pt
Model loaded


In [ ]:
import numpy as np
import os

os.makedirs('/content/embeddings', exist_ok=True)

protein_ids = list(sequences.keys())[:16]   # <-- only testing on 16 proteins first
batch_size = 8
max_len = 1022

all_embeddings = {}

for start in range(0, len(protein_ids), batch_size):
    batch_ids = protein_ids[start:start + batch_size]
    batch_data = [(pid, sequences[pid][:max_len]) for pid in batch_ids]

    _, _, batch_tokens = batch_converter(batch_data)
    batch_tokens = batch_tokens.to('cuda')

    with torch.no_grad():
        results = model(batch_tokens, repr_layers=[12])
        token_representations = results["representations"][12]

    for i, pid in enumerate(batch_ids):
        seq_len = len(batch_data[i][1])
        embedding = token_representations[i, 1:seq_len+1].mean(0).cpu().numpy()
        all_embeddings[pid] = embedding

print("Done with test batch")

Done with test batch


In [ ]:
print(len(all_embeddings))
example_id = list(all_embeddings.keys())[0]
print(all_embeddings[example_id].shape)
print(all_embeddings[example_id][:5])

16
(480,)
[-0.15716088 -0.02640764  0.23593917  0.17353383 -0.0369205 ]


In [ ]:
import numpy as np
import os

os.makedirs('/content/embeddings', exist_ok=True)

protein_ids = list(sequences.keys())   # full set this time
batch_size = 8
max_len = 1022

all_embeddings = {}

for start in range(0, len(protein_ids), batch_size):
    batch_ids = protein_ids[start:start + batch_size]
    batch_data = [(pid, sequences[pid][:max_len]) for pid in batch_ids]

    _, _, batch_tokens = batch_converter(batch_data)
    batch_tokens = batch_tokens.to('cuda')

    with torch.no_grad():
        results = model(batch_tokens, repr_layers=[12])
        token_representations = results["representations"][12]

    for i, pid in enumerate(batch_ids):
        seq_len = len(batch_data[i][1])
        embedding = token_representations[i, 1:seq_len+1].mean(0).cpu().numpy()
        all_embeddings[pid] = embedding

    if start % 800 == 0:
        print(f"Processed {start}/{len(protein_ids)}")
        np.save('/content/embeddings/checkpoint.npy', all_embeddings)

np.save('/content/embeddings/final_embeddings.npy', all_embeddings)
print("Done")

Processed 0/82404
Processed 800/82404
Processed 1600/82404
Processed 2400/82404
Processed 3200/82404
Processed 4000/82404
Processed 4800/82404
Processed 5600/82404
Processed 6400/82404
Processed 7200/82404
Processed 8000/82404
Processed 8800/82404
Processed 9600/82404
Processed 10400/82404
Processed 11200/82404
Processed 12000/82404
Processed 12800/82404
Processed 13600/82404
Processed 14400/82404
Processed 15200/82404
Processed 16000/82404
Processed 16800/82404
Processed 17600/82404
Processed 18400/82404
Processed 19200/82404
Processed 20000/82404
Processed 20800/82404
Processed 21600/82404
Processed 22400/82404
Processed 23200/82404
Processed 24000/82404
Processed 24800/82404
Processed 25600/82404
Processed 26400/82404
Processed 27200/82404
Processed 28000/82404
Processed 28800/82404
Processed 29600/82404
Processed 30400/82404
Processed 31200/82404
Processed 32000/82404
Processed 32800/82404
Processed 33600/82404
Processed 34400/82404
Processed 35200/82404
Processed 36000/82404
Proce

In [ ]:
print(len(all_embeddings))
print(len(protein_ids))

# sanity check: did every protein get an embedding, none missing or NaN?
missing = [pid for pid in protein_ids if pid not in all_embeddings]
print(f"Missing: {len(missing)}")

nan_check = sum(1 for emb in all_embeddings.values() if np.isnan(emb).any())
print(f"Embeddings with NaN: {nan_check}")


82404
82404
Missing: 0
Embeddings with NaN: 0


In [ ]:
import pickle

with open('/content/checkpoint_step3.pkl', 'wb') as f:
    pickle.dump({
        'embeddings': all_embeddings,
        'Y_filtered': Y_filtered,
        'kept_terms': kept_terms
    }, f)

print("Checkpoint saved")

Checkpoint saved


In [ ]:
X = np.array([all_embeddings[pid] for pid in protein_ids_ordered])

print(X.shape)
print(Y_filtered.shape)

(82404, 480)
(82404, 4828)


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.multioutput import MultiOutputClassifier
from sklearn.linear_model import LogisticRegression

X_train, X_val, Y_train, Y_val = train_test_split(
    X, Y_filtered, test_size=0.15, random_state=42
)

print(X_train.shape, X_val.shape)

(70043, 480) (12361, 480)


In [ ]:
with open('/content/checkpoint_before_training.pkl', 'wb') as f:
    pickle.dump({
        'X_train': X_train, 'X_val': X_val,
        'Y_train': Y_train, 'Y_val': Y_val
    }, f)
print("Saved")

Saved


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
shutil.copy('/content/checkpoint_step3.pkl', '/content/drive/MyDrive/checkpoint_step3.pkl')
shutil.copy('/content/checkpoint_before_training.pkl', '/content/drive/MyDrive/checkpoint_before_training.pkl')
print("Copied to Drive")

Mounted at /content/drive
Copied to Drive
